# 심층 신경망

In [2]:
import keras
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
from sklearn.model_selection import train_test_split

train_scaled = train_input / 255.0
train_scaled = train_scaled.reshape(-1, 28*28)
train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size=0.2, random_state=42)

### 은닉층
- 입력층과 출력층 사이에 있는 모든 층

### 활성화 함수
- **신경망 층의 선형 방정식의 계산 값에 적용**하는 함수
- 은닉층, 출력층 모두 활성화 함수를 가질 수 있음
  - 출력층 활성화 함수 예시 $\rightarrow$ 시그모이드 함수
  - 은닉층 활성화 함수 예시 $\rightarrow$ ReLU 함수
- 출력층의 활성화 함수는 제한되어 있음
  - 이진 분류: 시그모이드 함수
  - 다중 분류: 소프트맥스 함수
- 은닉층의 활성화 함수는 비교적 자유로움
  - 시그모이드 함수, ReLU 등..
- 회귀를 위한 신경망의 출력층 활성화 함수
  - 회귀의 출력은 임의의 어떤 숫자이므로 활성화 함수 적용할 필요 X
    - 출력층의 선형 방정식 계산값 그대로 출력
### 은닉층에 활성화 함수를 적용하는 이유
- 신경망에서 단순히 선형 계산(곱하기/더하기)만 하는 건 아무리 반복해도 한 번 계산한 것이랑 똑같음 -> **층이 깊어지는 효과가 사라짐**
  - Ex) 2를 곱하고 3을 곱하는 두 개의 층 -> 6 곱하는 층 하나랑 같음
- 이런 현상을 방지하기 위해 **선형 계산을 비선형적으로 비틀어줌**
  - **비선형 함수**(ReLU, 시그모이드 등)을 끼워넣음 -> 구부리거나 꺾는 역할
- 종이 접기로 이해하면 쉬움
  - 데이터가 평평한 종이라고 할 때, 직선 자로 이걸 나누는 선을 그리면 어떻게 그려도 결과는 직선
  - 자를 대고 선을 그은 뒤 종이를 구기거나 접은 후(비선형 변환) 다시 선을 그으면 직선이 V자 모양으로 꺾이거나 곡선으로 보임
    - 이걸 반복하면 원, S자, 복잡한 모양 등 어떤 복잡한 경계선을 만들어 낼 수 있음
- 요약하면, 활성화 함수는 **앞 층의 계산 결과와 뒷 층의 계산이 하나로 합쳐지는 걸 막고**, **직선을 곡선**으로 만들어 줌

In [ ]:
# 은닉층: 시그모이드 / 출력층: 소프트맥스
inputs = keras.layers.Input(shape=(784, )) # 입력층
dense1 = keras.layers.Dense(100, activation='sigmoid') # 뉴런 100개 가진 은닉층
dense2 = keras.layers.Dense(10, activation='softmax') # 뉴런 10개 가진 출력층

In [5]:
# 심층 신경망(Deep Neural Network, DNN) 만들기
model = keras.Sequential([inputs, dense1, dense2]) # 인자로 전달하는 리스트 -> 처음은 입력층, 마지막은 출력층, 그 사이에는 필요한 층 계속 추가
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

- 샘플마다 784개의 픽셀값이 은닉층을 통과하면서 100개의 특성으로 압축됨
- 파라미터 개수 계산법
  - Dense 층: 입력층 뉴런 784개 * 압축된 뉴런 100개 + 각 뉴런 절편 100개 = 78500개
  - 출력층: 이전 Dense 층 뉴런 100개 * 압축된 뉴런 10개 + 각 뉴런 절편 10개 = 1010개

In [6]:
# 층 더 추가하기

# 각 층을 따로 객체로 저장해 쓸 일이 거의 없음
# 주로 다음처럼 Sequential 클래스 생성자 안에서 바로 클래스 객체 만듦
model = keras.Sequential([
    keras.layers.Input(shape=(784, )),
    keras.layers.Dense(100, activation='sigmoid', name='은닉층'),
    keras.layers.Dense(10, activation='softmax', name='출력층')
], name='패션 MNIST 모델')

model.summary()

Model: "패션 MNIST 모델"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ 은닉층 (Dense)                  │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 출력층 (Dense)                  │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# add로 층 추가하기
model = keras.Sequential()
model.add(keras.layers.Input(shape=(784, )))
model.add(keras.layers.Dense(100, activation='sigmoid'))
model.add(keras.layers.Dense(50, activation='sigmoid'))
model.add(keras.layers.Dense(10, activation='softmax'))
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 10)             │           510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 84,060 (328.36 KB)

 Trainable params: 84,060 (328.36 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# 모델 훈련
model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_scaled, train_target, epochs=10)

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8799 - loss: 0.3335
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8861 - loss: 0.3101
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8877 - loss: 0.3052
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.8917 - loss: 0.2951
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8990 - loss: 0.2782
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9000 - loss: 0.2734
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9021 - loss: 0.2661
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9029 - loss: 0.2619
Epoch 9/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9067 - loss: 0.2554
Epoch 10/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9111 - loss: 0.2445


# 렐루 함수
- 초창기 인공 신경망의 은닉층에는 활성화 함수로 시그모이드 함수가 자주 쓰임
  - 오른쪽과 왼쪽 끝으로 갈수록 그래프가 누우므로 올바른 출력을 만드는데 신속하게 대응하지 못하는 단점 존재
  - 층이 많을 수록 그 효과가 누적되어 학습을 더 어렵게 만듦
    - 쉽게 말하면 **층이 많을수록 활성화 함수 양쪽 끝에서 변화가 작아서 학습 어려워짐**
  - 이를 개선하기 위해 제안된 것이 렐루(ReLU) 함수
- 렐루 함수
  - 입력이 양수일 경우 활성화 함수가 없는 것 처럼 입력 그대로 통과
  - 음수일 경우 0으로 만듦
  - $max(0, z)$로 쓸 수 있음
  - 이미지 처리에서 좋은 성능을 냄

In [ ]:
# Keras의 Flatten 층
# 배치 차원을 제외하고 나머지 입력 차원을 전부 1차원 일렬로 펼침
# 인공 신경망 성능에 기여하는 건 X. 단순히 펼치기만 함
# 입력층과 은닉층 사이에 추가하기 때문에 이를 층이라 부름 (실제 학습하는 층은 아님)

model = keras.Sequential()
model.add(keras.layers.Input(shape=(28, 28))) # 안 펼치고 28x28 원본 이미지 그대로 넣음
model.add(keras.layers.Flatten()) # 입력층과 은닉층 사이에 Flatten 층 추가 -> 1차원으로 펼침
model.add(keras.layers.Dense(100, activation='relu')) # 렐루 함수 사용
model.add(keras.layers.Dense(10, activation='softmax'))

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# reshape 메서드 적용하지 않고 훈련 데이터 다시 준비해서 모델 훈련시켜보기
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()
train_scaled = train_input / 255.0
train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size=0.2, random_state=42)

model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_scaled, train_target, epochs=5)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7701 - loss: 0.6686
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8596 - loss: 0.3967
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8697 - loss: 0.3568
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8782 - loss: 0.3331
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8863 - loss: 0.3190


In [17]:
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8753 - loss: 0.3535


[0.35813143849372864, 0.8743333220481873]

## 하이퍼파라미터
- 모델이 학습하지 않아 사람이 지정해야 하는 파라미터
- 지금까지 다룬 하이퍼파라미터 -> 추가할 은닉층 개수, 뉴런 개수, 활성화 함수, 층의 종류, 배치 사이즈, 에포크

## 옵티마이저
- 케라스에서 제공하는 다양한 종류의 경사 하강법 알고리즘
  - cf) 기본으로 RMSprop 사용

In [ ]:
# compile() 메서드의 매개변수를 변경해 옵티마이저 변경

# SGD 사용 (미니배치)
model.compile(optimizer='sgd', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 옵티마이저 객체로도 사용 가능
sgd = keras.optimizers.SGD()
model.compile(optimizer=sgd, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 학습률 변경
sgd = keras.optimizers.SGD(learning_rate=0.1) # 기본 학습률 0.01

# 다양한 옵티마이저들

## 기본 경사 하강법 옵티마이저
- 모두 SGD 클래스를 수정하면 됨
### 모멘텀 최적화(momentum optimization)
- 이전의 Gradient를 가속도처럼 사용할 수 있음
- momentum 매개변수 변경
### 네스트로프 모멘텀 최적화(nestrov momentum optimization)
- 모멘텀 최적화를 2번 반복해 구현
- 대부분의 경우 기본 확률적 경사 하강법보다 나은 성능 제공
- nestrov 매개변수 변경

## 적응적 학습률(adaptive learning rate) 옵티마이저
- 모델이 최적점에 가까이 갈수록 학습률을 낮춤 -> 안정적으로 최적점에 수렴할 가능성 증가
### RMSprop
### Adam
- 모멘텀 최적화 + RMSprop의 장점 합침
### Adagrad

In [ ]:
# 모멘텀 & 네스트로프 모멘텀 최적화
sgd = keras.optimizers.SGD(momentum=0.9, nestrov=True)

# Adagrad
adagrad = keras.optimizers.Adagrad()
model.compile(optimizer=adagrad, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# RMSprop
rmsprop = keras.optimizers.RMSprop()
model.compile(optimizer=rmsprop, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Adam
# RMSprop + Momentum Optimization
adam = keras.optimizers.Adam()
model.compile(optimizer=adam, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [18]:
# Adam 클래스의 매개변수 기본값으로 패션 MNIST 모델 훈련
model = keras.Sequential()
model.add(keras.layers.Input(shape=(28, 28)))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(100, activation='relu'))
model.add(keras.layers.Dense(10, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_scaled, train_target, epochs=5)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7668 - loss: 0.6757
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8540 - loss: 0.4067
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8678 - loss: 0.3638
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.8792 - loss: 0.3313
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8874 - loss: 0.3039


In [ ]:
model.evaluate(val_scaled, val_target) # RMSprop보다 조금 더 나은 성능

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8754 - loss: 0.3552


[0.3565962314605713, 0.871666669845581]